In [ ]:

# Cette version compare la question de l'utilisateur et les question prédéfinis, ensuite extraire la réponse 


import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

dans cette version je calcule ma similarité de la question d'utilisateur et les question que l'on a préparées ensuite trouver la réponse 

In [ ]:
def load_qa_data(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    questions, answers = [], []
    for entry in raw_data:
        for q in entry["questions"]:
            questions.append(q)
            answers.append(entry["answer"])
    return questions, answers


In [ ]:
def create_embeddings(questions, model_name='all-MiniLM-L12-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(questions)
    return model, embeddings


In [ ]:
def get_best_answer(user_question, questions, answers, embeddings, model, threshold=0.5):
    user_embedding = model.encode([user_question])[0]  # vector only
    similarities = np.dot(embeddings, user_embedding)  # produit scalaire avec tous les embeddings
    max_score = similarities.max()

    if max_score < threshold:
        return None, max_score

    best_index = similarities.argmax()
    return answers[best_index], max_score


In [ ]:
def get_answers_from_multiple_models(user_question, questions, answers):
    models = [
    "all-MiniLM-L12-v2",                                     
    "gtr-t5-base"                            
] 
    results = []
    
    for model_name in models:
        model, embeddings = create_embeddings(questions, model_name)
        best_answer, score = get_best_answer(user_question, questions, answers, embeddings, model)
        results.append({
            'model': model_name,
            'best_answer': best_answer,
            'score': score
        })
    
    return results

In [ ]:
json_path = "data.json"  

questions, answers = load_qa_data(json_path)

user_input = input("Pose ta question : ")

results = get_answers_from_multiple_models(user_input, questions, answers)

for result in results:
    print(f"\nModel: {result['model']}")
    if result['best_answer']:
        print(f" Réponse trouvée (score = {result['score']:.2f}): {result['best_answer']}")
    else:
        print(f"Aucune réponse pertinente trouvée (score = {result['score']:.2f})")


Model: all-MiniLM-L12-v2
❌ Aucune réponse pertinente trouvée (score = 0.39)

Model: gtr-t5-base
✅ Réponse trouvée (score = 0.71): Justificatif d’identité (C.N.I biométrique) (2x)
Acte de naissance (2x)
Justificatif de résidence (-3 mois) (2x)
Chèque CCP barré
Copie de la carte CHIFFA
Deux photos
Relevé de compte

Selon votre catégorie :
- Marié(e)s : Fiche familiale
- Salariés : 3 fiches de paie + Attestation de travail récente
- Retraités : Attestation + RENA
- Militaires : Présence au corps
- Retraité militaire : Attestation de radiation
- Si crédit en cours : copie de l’échéancier


: 